In [ ]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
OUT_DIR = ROOT / "data" / "output"
if not OUT_DIR.exists():
    OUT_DIR = ROOT.parent / "data" / "output"

parquet_path = OUT_DIR / "forecast.parquet"
if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

raw_df = pl.read_parquet(parquet_path)

In [ ]:
raw_df.head()

In [ ]:
df = raw_df.filter((pl.col("y") != 0)).with_columns(
    (pl.col("value") - pl.col("valuehat")).abs().alias("abs_err")
)["unique_id", "ds", "value", "valuehat", "abs_err", "period_type"]
print(df.head())

In [ ]:
sku_df = df.filter(
    (pl.col("unique_id") == "1||T:00001||S:46499")
    & (pl.col("period_type") == "in_sample")
)
print(sku_df.head())

In [ ]:
sku_df.count()

In [ ]:
print(sku_df["value", "valuehat", "abs_err"].sum())
print(192969.01 / 632903.05)

In [ ]:
wMAPE_sku = float(sku_df["abs_err"].sum()) / float(sku_df["value"].sum())
wMAPE_sku

In [ ]:
sto_df = df.filter(
    (pl.col("unique_id").str.contains("1||T:00001||S:", literal=True))
    & (pl.col("period_type") == "in_sample")
)
print(sto_df.head())

In [ ]:
print(sto_df["value", "valuehat", "abs_err"].sum())

In [ ]:
wMAPE_sto =  float(sto_df["abs_err"].sum()) / float(sto_df["value"].sum())
wMAPE_sto

In [ ]:
sec_df = df.filter(
    (pl.col("unique_id").str.count_matches(r"\|\|", literal=False) == 2)
    & (pl.col("period_type") == "in_sample")
)
print(sec_df.head())

In [ ]:
wMAPE_sec = float(sec_df["abs_err"].sum()) / float(sec_df["value"].sum())
wMAPE_sec